# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

In [1]:
%load_ext autoreload
%autoreload 2

import json
from _campaign_lib import *

# --- Services ---
svc = init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

# --- Campaign config ---
campaign_config = {
    "sample_size": 15,              # queries per eval step (service default: all)
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "exclude_steps": ["llm_ranking"],    # steps to skip (e.g. ["entity_profiling"])
    "pipeline_overrides": {},
    "optimization": {
        "patience": 2,                   # default: 3
        "max_rounds": None,              # default: 10 (None = unlimited, patience-only stop)
        "degradation_threshold": 0.4,    # fraction of degraded queries to trigger escalation (0 = disabled)
        "backend_warning_threshold": 2, # degradation resets before backend advisory (0 = disabled)
        "enable_l2": True,               # L2 refine_context on escalation
        "enable_l3": True,               # L3 modify_plan on L2 stall
        "l2_patience": None,             # default: 2 (None = unlimited L2 rounds)
        "l3_patience": None,             # default: 1 (None = unlimited L3 rounds)
    },
    "eval_llm": {
        # --- Groq (free tier, open-source models) ---
        "model": "openai/gpt-oss-120b",
        # "model": "moonshotai/kimi-k2-instruct-0905"
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",          # best quality
        # "model": "claude-sonnet-4-6",      # good balance
        # "model": "claude-haiku-4-5-20251001",  # cheapest
        # "provider_url": "https://api.anthropic.com",
        "max_tokens": 2000,              # response length budget
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                    "descriptions to standardized database terms using entity profiling "
                    "and candidate ranking.",
        "grid_budget": 35,               # default: 0 (full grid)
        "sample_size": 6,     # default: 0 (all queries)
        "shared_queries": False,          # default: True
    },
}

# --- Pipeline snapshot & params ---
pipeline_config_full = show_pipeline_snapshot(svc)
pipeline_params = configure_pipeline(svc, campaign_config)

# --- Data ---
EXCEL_PATH = r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx"  # e.g. "../data/BOM-example.xlsx"
FORCE_RELOAD = True  # Set True to re-read Excel and overwrite stored datasets

train_data, svc["session_terms"] = prepare_datasets(
    svc["store"], svc["backend_id"],
    excel_path=EXCEL_PATH or None,
    force=FORCE_RELOAD,
)

Backend: http://127.0.0.1:8000


2026-03-23 12:21:40 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"
2026-03-23 12:21:40 INFO     [api.services.pipeline_discovery] Matched known pipeline 'termnorm'; using enriched schema
2026-03-23 12:21:40 INFO     [api.services.campaign.campaign_init] Pipeline schema loaded: termnorm vv1.1
2026-03-23 12:21:40 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"


Pipeline: termnorm (6 steps)
Experiment: production_historical (40 queries, 93 session terms)
Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93
Loaded task description: 3751 chars from LCA_INPUT_PATTERNS.md
  PIPELINE SNAPSHOT: TermNorm v1.1
  Nodes:   ['fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking', 'direct_prompt']
  Schemas: ['entity_profile/1', 'llm_ranking_output/1']
  Prompts: ['entity_profiling/1', 'llm_ranking/1']

{
  "name": "TermNorm",
  "version": "v1.1",
  "available_models": [
    "moonshotai/kimi-k2-instruct-0905",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "moonshotai/kimi-k2-instruct",
    "openai/gpt-oss-120b"
  ],
  "nodes": {
    "fuzzy_matching": {
      "type": "DeterministicFunction",
      "config": {
        "threshold": 70,
        "scorer": "WRatio",
        "limit": 5
      }
    },
    "web_search": {
      "type": "ExternalService",


2026-03-23 12:21:41 INFO     [api.services.dataset_builder] Loaded 1231 ground-truth pairs from C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx (Sheet1: 823, Processing: 408)
2026-03-23 12:21:41 WARNING  [api.services.dataset_builder] Duplicate queries across test sets (20): ['Cu-ETP1 8bk/wire drawing', 'Blech Spaltband 73,2x1,00mm/sheet rolling', 'Polyethylen LDPE/Fully automatic process', 'Strip EN10140-0,8Bx28P GK-DC03+C390-MB CU3/stamping', 'Adhesive label 63x82 white\nPolyethylen (PE) weiß glänzend, Oberfläche drucklackiert, 80g/m2/stamping']
2026-03-23 12:21:41 INFO     [api.services.dataset_builder] Split: 984 train, 82 test_processes, 165 test_material



  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ------------------------------------------------
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets


In [2]:
#@title Task context decomposition
task_context = await decompose_task_context(TASK_DESCRIPTION, campaign_config, svc)

2026-03-23 12:21:45 WARNING  [langfuse] Prompt 'optimizer_restructure-label:production' not found during refresh, evicting from cache.


TASK CONTEXT DECOMPOSITION
  domain: Life Cycle Assessment (LCA) terminology normalization
  pipeline_purpose: Map free‑form material and process descriptors entered by users to exact entries in LCA databases (e.g., ecoinvent, GaBi) for downstream impact calculation.
  data_characteristics: Short textual inputs (5‑100 tokens) containing alphanumeric codes, symbols, mixed German/French/English, up to thousands of queries per day.
  optimization_goals: Increase exact‑match rate, improve confidence scoring, correctly flag no‑match cases, handle geographic variants, and reduce false positives from superficial string similarity.
  key_challenges: Unstructured shorthand, indirect standard references, brand‑specific names, composite materials, evolving synonyms, multilingual terms, and multiple geographic variants.

  Consultation: To boost profile schema quality, enrich the input schema with explicit sub‑fields (e.g., detected_code, brand, standard, geography, composite_parts) so the model c

In [3]:
#@title Prepare evaluation context
campaign_rounds = []
baseline_results = []

baseline_ps, eval_data, backend_status = prepare_eval_context(
    svc, train_data,
)

RUN_BASELINE = False  # Set True to evaluate baseline before exploration
if RUN_BASELINE:
    campaign_rounds, baseline_results = run_baseline_eval(
        baseline_ps, eval_data, campaign_config, svc,
    )


BACKEND STATUS
  Session Active                 False
  Active Sessions                0
  Terms Loaded                   0
  Match Database Identifiers     111
  Match Database Aliases         716
  Experiments Count              4
  Mappings Count                 1126
  Pipeline Version               v1.1
  Llm Provider                   groq
  Llm Model                      moonshotai/kimi-k2-instruct-0905
  ------------------------------------------------
  Experiments                   
    0_production_realtime        0 mappings
    1_production_historical      887 mappings
    2_bom_materials              159 mappings
    3_bom_processing             80 mappings

Evaluation data: 984 queries


In [4]:
#@title Experiment dashboard
# Set to a short hex ID (e.g. '68e2c5') to resume a specific experiment.
# The system adds prefixes (cycle_, scan_, etc.) per data type.
# Set to None to auto-detect from current campaign_config + eval_data.
EXPERIMENT_ID = '68e2c53845c3' #None
EXPERIMENT_ID = '77e7e77777e7' #None


# When EXPERIMENT_ID is set, load stored config → overrides notebook variables
if EXPERIMENT_ID:
    stored_cfg = load_experiment_config(svc["store"], svc["backend_id"], EXPERIMENT_ID)
    if stored_cfg:
        pp_override = apply_experiment_overrides(campaign_config, stored_cfg)
        if pp_override:
            pipeline_params = pp_override
        print(f"  Loaded config from experiment {EXPERIMENT_ID}")

show_experiment_dashboard(
    svc=svc, experiment_id=EXPERIMENT_ID,
    campaign_config=campaign_config, eval_data=eval_data,
    pipeline_params=locals().get("pipeline_params"),
    baseline_prompt_state=campaign_rounds[0]["prompt_state"].model_dump() if campaign_rounds else None,
)

2026-03-23 12:21:51 WARNING  [api.services.campaign.campaign_init] No campaign matching '77e7e77777e7'
2026-03-23 12:21:51 WARNING  [api.services.campaign.campaign_init] No campaign matching '77e7e77777e7'


## 3. Explore

Two exploration paths: **Smart Search** (scan advisor + sensitivity scan) or **Grid Search** (brute-force sweep). Use one or both.

In [5]:
#@title 3a. Smart Search — Browse variant library
# display_variant_library()
# Filter examples:
# display_variant_library(source="PromptWizard")
# display_variant_library(axes=["thinking_style", "persona"])

In [6]:
# preview_advisor_prompt()
preview_advisor_prompt(campaign_config, svc, task_description=task_context, raw=True)

2026-03-23 12:21:52 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


You are an expert prompt optimization advisor. Recommend which axes (parameters and prompt fields) to prioritize in a sensitivity scan.

## Constraints (apply strictly)
- Do NOT recommend *_model axes — place them in axes_to_skip.
- Response must fit within 1500 tokens. Be terse.

## Pipeline: TermNorm AI terminology normalization pipeline
Steps execute sequentially — each step's output feeds the next:
[
  {
    "name": "cache_lookup",
    "node_role": "cache",
    "short_circuit": true
  },
  {
    "name": "fuzzy_matching",
    "node_role": "candidate_source",
    "short_circuit": true
  },
  {
    "name": "web_search",
    "node_role": "enricher"
  },
  {
    "name": "entity_profiling",
    "node_role": "enricher"
  },
  {
    "name": "token_matching",
    "node_role": "candidate_source"
  }
]

## Task Context
- **domain**: Life Cycle Assessment (LCA) terminology normalization
- **pipeline_purpose**: Map free‑form material and process descriptors entered by users to exact entries in 

In [7]:
#@title Scan advisor
advisory, scan_variants, schema_labels = await run_scan_advisor(
    campaign_config, svc,
    task_description=locals().get("task_context") or locals().get("TASK_DESCRIPTION", ""),
)

2026-03-23 12:21:52 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


SCAN ADVISOR -- pipeline-aware sensitivity setup
  Pipeline: termnorm (v1.1)
  Steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking']
  Excluded: ['llm_ranking']
  Task context: Life Cycle Assessment (LCA) terminology normalization — Map free‑form material and process descriptors entered by us
  Calling openai/gpt-oss-120b ...



2026-03-23 12:21:57 WARNING  [api.services.search.scan_advisor] Scan advisor validation: pipeline_param axis 'entity_profiling_schema_add_geo' not found in PipelineSchema param_keys: ['content_char_limit', 'fuzzy_scorer', 'fuzzy_threshold', 'max_sites', 'max_token_candidates', 'num_results', 'profiling_max_tokens', 'profiling_model', 'profiling_prompt', 'profiling_schema', 'profiling_temperature', 'query_prefix', 'query_suffix', 'ranking_max_tokens', 'ranking_model', 'ranking_prompt', 'ranking_sample_size', 'ranking_schema', 'ranking_temperature', 'raw_content_limit', 'relevance_weight_core']


----------------------------------------------------------------------
PRIORITY AXES (ranked by importance)
----------------------------------------------------------------------
  1. [HIGH] fuzzy_scorer (pipeline_param) -- step: fuzzy_matching
     Key to capture variant spellings
     Values: ['ratio', 'partial_ratio', 'token_set_ratio']
  2. [HIGH] fuzzy_threshold (pipeline_param) -- step: fuzzy_matching
     Controls match strictness
     Values: ['85', '90', '95']
  3. [MEDIUM] query_prefix (pipeline_param) -- step: web_search
     Guides retrieval toward LCA sources
     Values: ['LCA material', 'ecoinvent']
  4. [MEDIUM] query_suffix (pipeline_param) -- step: web_search
     Filters to domain‑specific sites
     Values: ['site:ecoinvent.org', 'site:gaebi.com']
  5. [MEDIUM] profiling_max_tokens (pipeline_param) -- step: entity_profiling
     Allows richer entity extraction
     Values: ['256', '512']
  6. [HIGH] entity_profiling_schema_add_geo (pipeline_param) -- step: entity_pr

In [8]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_sample_size = 10  # queries per scan variant (0 = use all)

scan_variants = {
    'max_token_candidates': [10, 30, 50],
    'query_prefix': ['what material is', 'identify LCA database name for', 'translate trade name'],
    'profiling_schema': [
        [['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'], ['+', 'database_format_hint', 'string', False, "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'"]],
        # [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_synonyms', 'array', False, 'Terms likely to appear verbatim in LCA database entry names for this entity'], ['+', 'no_match_signal', 'string', False, 'Brief reasoning on whether a database match is likely to exist or not']],
        # [['~', 'classification_aliases', 'lca_classification_aliases', 'array', False, 'Expert-level aliases specifically aligned with LCA database naming conventions, including ecoinvent activity names and SimaPro process names'], ['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA']],
        [['+', 'lca_database_names', 'array', True, "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'"]], 
        [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions']],
        [['~', 'notes', 'material_category', 'string', True, "The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'"]]
    ],
    'profiling_temperature': [0.0, 0.3, 0.7],
    # 'profiling_max_tokens': [512, 1024, 2048], # -> Going to cause lots of Errors.
    'raw_content_limit': [1000, 2500, 8000],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

  max_token_candidates: [10, 30, 50]
  query_prefix: ['what material is', 'identify LCA database name for', 'translate trade name']
  profiling_schema: (baseline + 4 mutations)
    [0] (baseline)
    [1] ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'), ('+', 'database_format_hint', 'string', False, 'Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'')
    [2] ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'')
    [3] ('-', 'manufacturing_processes'), ('-', 'applications'), ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions')
    [4] ('~', 'notes', 'material_category', 'string', True, 'The broad LCA material 

In [9]:
#@title Prepare scan baseline
# Scan always uses fresh pipeline defaults (not experiment overrides) so the
# baseline content hash matches previous runs regardless of EXPERIMENT_ID.
scan_pipeline_params = configure_pipeline(svc, campaign_config)
scan_baseline_sp, scan_coverage = await prepare_scan_baseline(
    baseline_ps, campaign_config,
    pipeline_params=scan_pipeline_params,
    svc=svc, scan_variants=scan_variants,
)

2026-03-23 12:21:57 INFO     [api.services.search.context] restructure_context_cached: hit (alias group)


Active steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching']
  Excluded: ['llm_ranking']
  Restructured baseline fields (cached):
    persona: You are a candidate evaluation expert.
    task_intent: Analyze an entity profile, extract its category and distinguishing features, the...
    problem_description: Given a JSON entity profile and a list of candidate strings, the model must iden...
    instruction: TASK 1: Summarize the profile in 1‑2 sentences, identify the entity_category, an...
    thinking_style: Think step by step.
    answer_format: JSON with fields 'reasoning' (string) and 'ranked_candidates' (array of objects ...
  Search baseline: 5f9866616eda (render: 1266 chars)


2026-03-23 12:21:57 INFO     [api.services.search.coverage] build_prompt_result_index: 1 runs -> 1 unique prompts, 10 total query results



  Historical data: 10 results across 1 unique prompts
  Matching runs (sp_hash): 1, 10 cached results

  Scan variant coverage (1 matching runs):
    max_token_candidates     (not tested)
    query_prefix             (not tested)
    profiling_schema         (not tested)
    profiling_temperature    (not tested)
    raw_content_limit        (not tested)


In [ ]:
#@title Sensitivity scan
scan_df, axis_profiles = await sensitivity_scan(
    scan_baseline_sp, scan_variants, eval_data,
    sample_size=locals().get('scan_sample_size', 10),
    svc=svc, experiment_id=EXPERIMENT_ID or "",
)

In [11]:
# #@title Scan analytics (uncomment to display)
# if scan_df is not None and not scan_df.empty:
#     show_scan_leaderboard(scan_df, axis_profiles)
#     difficulty_df = show_scan_query_difficulty(svc["store"], svc["backend_id"])

In [12]:
#@title Select scan winner & seed campaign
best_sp = seed_campaign_from_scan(
    scan_df, axis_profiles, scan_baseline_sp, scan_variants,
    campaign_rounds, campaign_config,
)

No scan data available. Run sensitivity scan first.
Updated pipeline_params: {'steps': ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching']}


TypeError: 'NoneType' object is not subscriptable

### 3b. Grid Search

<details>
<summary>Grid search cells (click to expand)</summary>

Systematic sweep of the prompt configuration space. Maps the accuracy landscape before hill-climbing. All cells below are commented out by default.

**To activate:** uncomment cells below and run in order. Grid search evaluates all combinations of prompt fields and pipeline params — expect 100–500+ backend calls depending on `grid_budget` and `sample_size` in `campaign_config["grid_search"]`.

</details>

In [ ]:
# #@title Grid campaign overview (existing plans)
# merge_plans = False  # Set True to combine results from multiple plans
# grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
# merged_grid_df = grid_overview.get("merged_grid_df")

In [ ]:
# #@title Build or resume grid plan
# gs = campaign_config["grid_search"]

# llm_client, llm_model = setup_llm(campaign_config)

# (
#     grid_plan_id, grid_points, grid_state_lookup,
#     grid_axes, layer1_fields, grid_baseline,
# ) = await resume_or_build_grid(
#     campaign_config, baseline, llm_client, llm_model,
#     svc["store"], svc["backend_id"],
#     improvement_areas=campaign_config.get("improvement_areas", ""),
# )

# print(f"Grid points: {len(grid_points)}")
# print(f"Plan ID: {grid_plan_id}")

In [ ]:
# #@title Run grid search
# grid_df = await run_grid_search(
#     grid_points, grid_state_lookup, eval_data,
#     campaign_config["eval_llm"],
#     plan_id=grid_plan_id,
#     svc=svc,
#     pipeline_params=campaign_config.get("pipeline_params"),
#     sample_size=gs.get("sample_size", 1),
#     shared_queries=gs.get("shared_queries", False),
#     grid_seed=gs.get("seed", 42),
# )

In [ ]:
# #@title Display grid results
# _display_df = merged_grid_df if merged_grid_df is not None else grid_df
# display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [ ]:
# #@title LLM analysis of grid results
# _analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
# llm_client, llm_model = setup_llm(campaign_config)
# grid_analysis = await analyze_grid_results(
#     _analysis_df, grid_axes, llm_client, model=llm_model,
# )

In [ ]:
# #@title Select grid winner and seed campaign
# grid_winner = select_and_seed_grid_winner(
#     grid_df, merged_grid_df, grid_state_lookup,
#     grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
# )

## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [ ]:
#@title Feedback cycle preflight
scan_context = show_feedback_preflight(
    campaign_rounds, eval_data, campaign_config,
    pipeline_params=pipeline_params,
    scan_df=locals().get("scan_df"),
    axis_profiles=locals().get("axis_profiles"),
    scan_variants=locals().get("scan_variants"),
    difficulty_df=locals().get("difficulty_df"),
)


  FEEDBACK CYCLE PRE-FLIGHT
  Baseline accuracy      : 10.0%
  Baseline prompt        : TASK 1: Summarize the profile in 1‑2 sentences, identify entity_category, and li...
  ------------------------------------------------------------------
  Max rounds             : 3
  Candidates per round   : 5
  Queries per eval       : 15 of 984
  Improvement threshold  : 1.0%
  Patience (L1)          : 2 rounds
  L2 (refine context)    : enabled, patience=None
  L3 (modify plan)       : enabled, patience=None
  ------------------------------------------------------------------
  Candidate model        : openai/gpt-oss-120b
  Creativity             : 0.7
  Pipeline               : 5 of 6 steps
    Steps                : cache_lookup, fuzzy_matching, web_search, entity_profiling, token_matching
    Excluded             : llm_ranking
  Strategy               : SCAN-AWARE

  ROUND PIPELINE (what happens each round)
  ------------------------------------------------------------------
  1. BASELINE IN

In [ ]:
#@title Run optimization (feedback cycle)
# Force-reload api modules (ensures code edits take effect without kernel restart)
import importlib, sys
for _m in [
    "api.services.campaign.escalation",
    "api.services.campaign.layer_transitions",
    "api.services.campaign.critique",
    "api.services.campaign.models",
    "api.services.prompt_optimizer",
    "api.nodes.optimizer_nodes",
    "api.services.campaign.feedback_cycle",
]:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    svc=svc,
    pipeline_params=pipeline_params,
    scan_context=locals().get("scan_context"),
    experiment_id=locals().get("EXPERIMENT_ID"),
    task_context=locals().get("task_context"),
)

  Using stored baseline 20.0% (notebook had 10.0%)
  Interrupt of cells can take up to 60 seconds!
  If a dialog pops up, click 'Cancel' and wait 20 seconds.

╔════════════════════════════════════════════════════════════════════╗
║  FEEDBACK CYCLE STARTING                                           ║
╠════════════════════════════════════════════════════════════════════╣
║  Baseline       20.0%                                              ║
║  Max rounds     3              Patience    2                       ║
║  Candidates     5                                                  ║


2026-03-20 18:58:08 INFO     [api.services.campaign.feedback_cycle] Using provided baseline (acc=0.200)
2026-03-20 18:58:08 INFO     [api.services.campaign.feedback_cycle] Cycle identity: cycle_68e2c53845c3
2026-03-20 18:58:08 INFO     [api.services.campaign.feedback_cycle] Resuming cycle cycle_68e2c53845c3 — 2 prior round(s) on disk


║  Sample size    15 of 984                                          ║
║  Min detectable ±36.2% (α=0.05, 80% power)                         ║
║  Model          openai/gpt-oss-120b                                ║
║  L2 (refine)    enabled            L3 (plan)   enabled             ║
║  Scan context   YES                                                ║
║  Critique       enabled                                            ║
╚════════════════════════════════════════════════════════════════════╝


2026-03-20 18:58:08 INFO     [api.services.obs.observability_logger] Dataset 'termnorm_ground_truth': 728 items registered, 256 duplicates/empty skipped (from 984 input)
2026-03-20 18:58:08 WARNING  [api.services.obs.observability_logger] Skipping Langfuse cloud dataset registration for 984 items (rate-limit risk). Use the dedicated Langfuse sync cell instead.
2026-03-20 18:58:08 INFO     [api.services.campaign.feedback_cycle] Registered 728 dataset items for 'termnorm_ground_truth'
2026-03-20 18:58:08 INFO     [api.services.campaign.feedback_cycle] Restored optimizer state from round 1 (critique=1401 chars, task_context=0 keys, escalation_journal=0 entries, l2_round=0)
2026-03-20 18:58:08 INFO     [api.services.campaign.feedback_cycle] Registered prompt alias: d3cbb647 ↔ 41f88bae
2026-03-20 18:58:08 INFO     [api.services.campaign.feedback_cycle] Feedback cycle round 0 (clean=0/3, acc=0.200, stall=0/2)
2026-03-20 18:58:08 INFO     [api.services.campaign.feedback_cycle] Loaded 5 persis

  ✓ Initialized  cycle=cycle_68e2c5  samples=15  obs=ON
    Resumed from round 2 (2 rounds cached)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ROUND 1/3                                                 patience 0/2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

├─ GENERATE ─────────────────────────────────────────────────────────────┤
│  Current best    20.0%
│  Prompt          You are a candidate evaluation expert.  Summari...
│  Candidates      5   Creativity: 0.7   Scan: YES   Critique: YES
│  Model           openai/gpt-oss-120b
│  Scan focus: 4 improving axes [max_token_candidates, query_prefix, profiling_schema, profiling_temperature]
│  Scan baseline: 10.0%
├────────────────────────────────────────────────────────────────────────┤
  ✓ 5 candidates generated (loaded from disk)
    C1: Increase max_token_candidates to 40 and set pro...
    C2: Swap query_prefix to a semantic variant and exp...
    C3: Raise raw_content_l

...

  ┌─ SCOREBOARD ───────────────────────────────────────────────────────────────┐
  │  #   Label    Accuracy            95% CI  Composite    Delta               │
  │  1   C1         13.3%       [3.7%-37.9%]     0.1633     -6.7%  *           │
  └────────────────────────────────────────────────────────────────────────────┘
  ⚠ NO IMPROVEMENT  best candidate 20.0%  composite=0.2267
  Critique: Strengths: The token‑matching step does retrieve the correct LCA entry in many cases – the ground‑truth appears in the candidate list for 11/12 misses and ranks within the top‑10 for 60% of queries. Using the prefix "what material is" and a candidate pool of 30 tokens yields the highest observed accuracy (30%). The 12‑field profiling schema also outperforms smaller schemas. Weaknesses: Overall top‑1 accuracy is only 20% and many correct entries are buried deep (ranks 13‑19). The model frequently confuses material type with the manufacturing process (e.g., predicts "Acrylonitrile‑butadiene‑styrene" 

2026-03-20 18:58:14 WARNING  [langfuse] Prompt 'optimizer_l2_refine_context-label:production' not found during refresh, evicting from cache.
2026-03-20 18:58:15 INFO     [api.services.campaign.layer_transitions] L2 refine_context: 3 param changes, task_context updated, action=probe, directive=240 chars
2026-03-20 18:58:15 INFO     [api.services.campaign.feedback_cycle] L2 requested probe — next round uses warned queries
2026-03-20 18:58:15 INFO     [api.services.campaign.feedback_cycle] L2 refine_context at round 1 (l2_round=1)
2026-03-20 18:58:15 INFO     [api.services.stores.campaign_store] Deleted cached candidates for round 2 (escalation invalidation)
2026-03-20 18:58:15 INFO     [api.services.campaign.feedback_cycle] PROBE round 2: 11 warned queries (from 6 tracked)
2026-03-20 18:58:15 INFO     [api.services.campaign.feedback_cycle] Feedback cycle round 2 (clean=1/3, acc=0.200, stall=0/2, PROBE)


  ✓ L2 decision: 3 param changes, task_context updated, action=probe
    L2: Process bias and larger candidate pool directly address the main failure mode of
    ⚠ 6 queries with recurring pipeline warnings (web_search:partial_scrape)

  --- L2 PROMPT (sent to LLM) ---
  │ You are a prompt optimization expert.
  │ 
  │ The L1 inner optimization loop has stalled — candidates are no longer improving.
  │ 
  │ ROUND HISTORY (stalled):
  │   Round 0: acc=20.0%
  │   Round 1: acc=20.0%
  │ 
  │ CURRENT PROMPT:
  │ ---
  │ You are a candidate evaluation expert.
  │ 
  │ Summarize the entity profile, identify its category and key distinguishing features, then score and rank 20 candidate matches against a core concept.
  │ 
  │ Given an entity_profile_json and a list of candidate matches, produce a concise reasoning summary, assign relevance_score (0.0‑1.0) per defined bands, and output a ranked list respecting score order.
  │ 
  │ TASK 1: Summarize the profile in 1‑2 sentences, identify enti

2026-03-20 18:58:17 WARNING  [langfuse] Prompt 'optimizer_meta_scan_aware-label:production' not found during refresh, evicting from cache.
2026-03-20 18:58:22 INFO     [api.services.stores.campaign_store] Saved 5 candidates for round 2 → round_0002_candidates.json


  ✓ 5 candidates generated (from LLM)
    C1: Switch query prefix to task‑specific phrase and... [instruction]
    C2: Increase candidate token pool to 45 to capture ... [instruction]
    C3: Set profiling temperature to a mid‑range value ... [instruction]
    C4: Introduce explicit ranking bias toward process ... [instruction]
    C5: Raise raw content limit to 12000 to give web se... [instruction]

│  Settings diff (18 params, 7 SPs):
│                                             Start   Parent  C1      C2      C3      C4      C5      
│  ─── web_search ────────────────────────────
│                               query_prefix  [a]     [b]     [c]     ·       ·       ·       ·       
│  ─── entity_profiling ──────────────────────
│                           profiling_schema  -       [d]     [e]     ·       ·       ·       ·       
│         profiling_schema.alternative_names  array   -       -       -       -       -       -       
│              profiling_schema.applications  [f]    

In [ ]:
#@title 5. Results — Campaign comparison, flip tracking, lineage
show_campaign_summary(campaign_rounds)
show_flip_tracking(campaign_rounds)
show_lineage_chain(campaign_rounds)

In [ ]:
#@title Save winner
save_campaign_winner(
    campaign_rounds, campaign_config, svc["store"], svc["backend_id"],
    experiment_id=locals().get("EXPERIMENT_ID"),
)

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print("--- SUGGESTED CONFIG (copy to Setup) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

In [ ]:
#@title Sync evaluation history to Langfuse
# Safe to re-run — already-pushed runs are skipped automatically.
stats = sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)